In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Dict, Type, Optional
from semantic_chunker import chunk_text_by_alpha
import json


In [2]:
CLIENT = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="none")

## 1. Coreference resolution

In [3]:
with open("prompts/coreference_resolution.txt", "r") as f:
    SYSTEM_PROMPT = f.read()

In [4]:
class CoreferenceResolutionResult(BaseModel):
    scratchpad: str = Field(
        description="scratchpad for LLM to identify pronouns, pointers like \"यह\", \"इस\", \"उस\", \"इन\", etc. and vague entities and identify which entities will replace them."
    )
    resolved_sentences: List[str] = Field(
        description="Simplified, modern Khadi Boli sentences with all pronouns, pointers like \"यह\", \"इस\", \"उस\", \"इन\", etc. and vague entities resolved."
    )

In [5]:
def parse_thought_and_response(full_text):
    """
    Helper function to separate the model's internal reasoning from the final answer.
    """
    if "<|channel>thought" in full_text and "<channel|>" in full_text:
        # Split the text around the closing tag
        parts = full_text.split("<channel|>")
        
        # Clean up the thought process (removing the opening tag)
        thought_process = parts[0].replace("<|channel>thought", "").strip()
        
        # The final answer is everything after the closing tag
        final_answer = parts[1].strip()
        
        return {
            "thought_process": thought_process,
            "final_answer": final_answer,
            "raw_output": full_text
        }
    
    # Fallback if the model didn't output thought tags for some reason
    return {
        "thought_process": None,
        "final_answer": full_text.strip(),
        "raw_output": full_text
    }

In [6]:
def generate_with_thinking(
    system_prompt: str, 
    user_query: str, 
    model_name: str = "Qwen/Qwen2.5-14B-Instruct",
    response_model: Optional[Type[BaseModel]] = None, # Defaults to None
    enable_thinking: bool = False
):
    # Enable thinking by injecting the token at the start of the system instructions
    system_prompt = f"<|think|>\n{system_prompt}" if enable_thinking else system_prompt

    # 1. Define the base parameters that are always required
    api_params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 11000, 
        "temperature": 0.2,
    }

    # 2. Conditionally inject the response_format only if a model is provided
    if response_model is not None:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema()
            }
        }

    # 3. Unpack the dictionary into the API call using **
    response = CLIENT.chat.completions.create(**api_params)
    
    full_response = response.choices[0].message.content
    return parse_thought_and_response(full_response)

In [26]:
for i in range(1):
    with open(f"text_files/chapter_{i}.txt", "r") as f:
        text = f.read()
        result = chunk_text_by_alpha(text)
        print(json.dumps(result['chunks'], ensure_ascii=False, indent=4))

Calculated Threshold (Average): 0.5417 (Min: 0.3239, Max: 0.7863, Avg: 0.5417)
[
    "संवत्सरारस्भ: इसमें सूर्योद्य- व्यापिनी प्रतिपदा लेनी चाहिए।  दोनों दिन सूर्योदय में प्रतिपदा हो या दोनों ही दिन सूर्योदय में प्रतिपदा न हो तो पहले ' दिन ही करना चाहिए।  यदि अधिक मास आ जावे तो भी प्रथम चैत्र शुक्ल प्रतिपदा को ही संवत्सरारम्भ मानना चाहिए; क्योंकि ऐसा अधिक मास अगले वर्ष में ही गिना जाता है।  इस दिन घरों पर ध्वजा लगाना, पञ्चाङ्ग- श्रवण, तैलाभ्यङ्ग और मिश्री तथा काली मिर्च- सहित नीम के पत्ते खाये जाते हैं।  पञ्चाङ्गों में जो श्लोक लिखे रहते हैं उनमें नीम के पत्ते के साथ मिश्री के स्थान पर नमक' और हींग, जीरा तथा अजवायन लिखे हैं।  तैलाभ्यङ्ग इस दिन अनिवार्य माना जाता है।  धर्मशास्त्रों में इस दिन महाशान्ति करने का और ब्रह्माजी के एवं वर्ष, मास, ऋतु, पक्ष, दिवस आदि कालावयवों के पूजन का भी विधान है।  इस दिन आरोग्यव्रत और तिलकव्रत भी किये जाते हैं।  ऊपर लिखा जा चुका है कि ऋतुओं के परिवर्त्त का नाम संवत्सर है।  अब देखना यह है कि इस चक्र का आरम्भ कब से होना चाहिए, क्योंकि जो गोल या चक्र के आकार 

In [31]:
import json

for i in range(1):
    # Added encoding="utf-8" which is crucial for handling Hindi/Sanskrit characters safely
    with open(f"text_files/chapter_{i}.txt", "r", encoding="utf-8") as f:
        chapter_text = f.read()
    
    print(f"Processing Chapter {i}: Coreference Resolution...")

    # 1. Coreference Resolution First (on the entire chapter)
    result = generate_with_thinking(
        SYSTEM_PROMPT + "\nText:\n", 
        chapter_text, 
        # model_name="google/gemma-3-27b-it",
        response_model=CoreferenceResolutionResult, 
        enable_thinking=False
    )
    
    parsed_result = CoreferenceResolutionResult.model_validate_json(result['final_answer'])
    resolved_data = parsed_result.model_dump()
    resolved_sentences = resolved_data['resolved_sentences']
    print(json.dumps(resolved_sentences, indent=4, ensure_ascii=False))
    
    # Stitch the resolved sentences back into a continuous string for the chunker
    fully_resolved_text = " ".join(resolved_sentences)

    print(f"Processing Chapter {i}: Semantic Chunking...")

    # 2. Semantic Chunking Second (on the resolved Khadi Boli/Hindi text)
    chunking_result = chunk_text_by_alpha(fully_resolved_text) 
    chunks = chunking_result['chunks']
    print(json.dumps(chunks, indent=4, ensure_ascii=False))
    break
    
    # 3. Output Generation
    # Saving both the chunks and the raw resolved sentences for traceability
    # output_data = {
    #     "chapter_id": i,
    #     # "resolved_sentences": resolved_sentences,
    #     "semantic_chunks": chunks
    # }
    
    # json_output = json.dumps(output_data, ensure_ascii=False, indent=4)
    # with open(f"coreference_output_files/chapter_{i}_output.json", "w", encoding="utf-8") as f:
    #     f.write(json_output)
        
    # print(f"Chapter {i} saved successfully.\n")

Processing Chapter 0: Coreference Resolution...
[
    "संवत्सरारम्भ में सूर्योद्य-व्यापिनी प्रतिपदा लेनी चाहिए।",
    "यदि दोनों दिन सूर्योदय में प्रतिपदा हो या दोनों ही दिन सूर्योदय में प्रतिपदा न हो तो पहले दिन ही करना चाहिए।",
    "यदि अधिक मास आ जाए तो भी प्रथम चैत्र शुक्ल प्रतिपदा को ही संवत्सरारम्भ मानना चाहिए;",
    "इस दिन घरों पर ध्वजा लगाना, पञ्चाङ्ग-श्रवण, तैलाभ्यङ्ग और मिश्री तथा काली मिर्च-सहित नीम के पत्ते खाये जाते हैं।",
    "धर्मशास्त्रों में इस दिन महाशान्ति करने का और ब्रह्माजी के एवं वर्ष, मास, ऋतु, पक्ष, दिवस आदि कालावयवों के पूजन का भी विधान है।",
    "इस दिन आरोग्यव्रत और तिलकव्रत भी किये जाते हैं।",
    "ऋतुओं के परिवर्त्त का नाम संवत्सर है।",
    "चैत्रशुक्ल प्रतिपदा को ही संवत्सर का आरम्भ होता है।",
    "ब्रह्माजी ने चैत्रमास में शुक्ल पक्ष के प्रथम दिन सूर्योदय होने पर जगत् की सृष्टि की है।",
    "इस दिन संवत्सरारम्भ का दिन कहते हैं।",
    "वसन्त ऋतु नवीन पत्र-पुष्पों द्वारा प्रकृति के नव शृङ्गार का आरम्भ-समय है।",
    "सूर्य, निरयन पक्ष के अनुसार और सायन पक्

## Entity Extraction

In [7]:
# 1. The Inner Model (Represents a single sentence and its entities)
class SentenceExtraction(BaseModel):
    sentence: str = Field(
        description="The simplified and resolved Hindi sentence."
    )
    entities: List[str] = Field(
        description="A list of extracted root-form entities (nodes) from the sentence. Must be strings."
    )

# 2. The Outer Model (Represents the final JSON payload)
class ExtractedNodes(BaseModel):
    extracted_entities: List[SentenceExtraction] = Field(
        description="A list of objects mapping sentences to their extracted entities."
    )

In [8]:
with open("prompts/entity_extraction_prompt.txt", "r", encoding="utf-8") as f:
    ENTITY_EXTRACTION_PROMPT = f.read()

In [9]:
for i in range(25):
    full_entity_extraction_result = []
    with open(f"coreference_output_files/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        coref_data = json.load(f)
        # print(json.dumps(coref_data, indent=4, ensure_ascii=False))
        for item in coref_data:
            user_query = json.dumps(item, indent=4, ensure_ascii=False)
            result = generate_with_thinking(ENTITY_EXTRACTION_PROMPT + "\n\n", user_query, response_model=ExtractedNodes, enable_thinking=True)
            parsed_result = ExtractedNodes.model_validate_json(result['final_answer'])
            entities = parsed_result.model_dump()
            full_entity_extraction_result.append(entities)
    
    json_output = json.dumps(full_entity_extraction_result, indent=4, ensure_ascii=False)
    # print(json_output)
    with open(f"entity_output_files_redone/chapter_{i}_output.json", "w", encoding="utf-8") as f:
        f.write(json_output)
        # user_query = json.dumps(coref_data, indent=4, ensure_ascii=False)
        # result = generate_with_thinking(ENTITY_EXTRACTION_PROMPT + "\n\n", user_query, response_model=ExtractedNodes, enable_thinking=True)
        # parsed_result = ExtractedNodes.model_validate_json(result['final_answer'])
        # json_output = parsed_result.model_dump_json(indent=4, ensure_ascii=False)
        # with open(f"entity_output_files/chapter_{i}_output.json", "w", encoding="utf-8") as f:
        #     f.write(json_output)

1. json_files/entity_info.json aur json_files/entity_id_dict.json uthana hai.
2. chapter 0 to 9 tak me entity_output_files aur entity_output_files_redone folder me se f"chapter_{i}_output.json" read karke ek ek entity ka 1 sentence ka description generate karana hai. context k liye pura chapter ka text dena hai.
3. json_files/entity_info.json me har entity k object me description key me ye add kar dena hai.

In [3]:
import json
from collections import defaultdict
from openai import OpenAI

In [4]:
CLIENT = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="none")

In [5]:
def call_llm(system_prompt, user_query, model_name="Qwen/Qwen2.5-14B-Instruct", max_tokens=8192, response_model=None):
    api_params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content

In [16]:
entt_to_sent = defaultdict(set)

with open("json_files/entity_info.json", "r", encoding="utf-8") as f:
    entity_repo = json.load(f)

with open("json_files/entity_id_dict.json", "r", encoding="utf-8") as f:
    entt_to_id_dict = json.load(f)

In [20]:
for i in range(10):
    with open(f"entity_output_files/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        old_file = json.load(f)
    with open(f"entity_output_files_redone/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        new_file = json.load(f)    
    
    for grp in old_file:
        for obj in grp['extracted_entities']:
              sentence = obj['sentence']
              entities = obj['entities']
              for entt in entities:
                  entt_to_sent[entt].add(sentence)
    
    for grp in new_file:
        for obj in grp['extracted_entities']:
              sentence = obj['sentence']
              entities = obj['entities']
              for entt in entities:
                  entt_to_sent[entt].add(sentence)
    

In [ ]:
import json
from collections import defaultdict
from pydantic import BaseModel, Field
from typing import Optional

# Assume CLIENT is already defined (e.g., from openai import OpenAI)
# and call_llm function is available as shown in the prompt.

class EntityDescription(BaseModel):
    scratchpad: str = Field(
        description="Think how would you explain this is entity to someone who is seeing this entity for the first time."
    )
    description: str = Field(
        description="Final general purpose description/definition/meaning of the entity"
    )

def main():
    # Load existing entity info and id mapping
    with open("json_files/entity_info.json", "r", encoding="utf-8") as f:
        entity_info = json.load(f)

    with open("json_files/entity_id_dict.json", "r", encoding="utf-8") as f:
        entity_to_id = json.load(f)  # not directly used, but kept for completeness

    processed_entities = set()

    # Optional: load existing descriptions to avoid reprocessing
    for ent, data in entity_info.items():
        if data.get("description"):
            processed_entities.add(ent)

    # System prompt for the LLM
    system_prompt = """
You are generating dictionary-style entity descriptions from a Hindi book.

Your task is to identify the GENERAL meaning, role, identity, or category of the entity — not the specific event, ritual instance, sentence context, or chapter-specific occurrence.

Write exactly ONE concise sentence in Hindi (Devanagari script).

Guidelines:
1. Describe what the entity fundamentally is.
2. Prefer broad, reusable definitions over context-specific descriptions.
3. If the entity is:
   - a festival/ritual -> describe the ritual generally
   - a person/deity -> describe who they are generally
   - a text/scripture -> describe its purpose or nature
   - a time/unit/date -> describe the calendrical concept generally
   - an object/substance -> describe what it is generally
   - an action/process -> describe the action generally
4. Use chapter context only to disambiguate meaning, not to copy chapter details.
5. Avoid mentioning:
   - specific incidents
   - one particular ritual occurrence
   - chapter events
   - temporary usage in a sentence
   - examples unless absolutely necessary
   - detailed explanations
   - lists of actions
6. Do NOT define an entity using another very specific sentence from the chapter.
7. Do NOT include phrases like:
   - "इस दिन..."

   unless the entity itself is specifically that event/date.
8. Keep descriptions self-contained and generally valid across contexts.

Good examples:
- "विधान धार्मिक कार्यों को सम्पन्न करने की निर्धारित प्रक्रिया या नियम है।"
- "पञ्चाङ्ग हिन्दू कालगणना और ज्योतिषीय गणना का ग्रन्थ है।"
- "नीम एक औषधीय गुणों वाला वृक्ष है।"

Bad examples:
- "विधान में नीम के पत्ते खाना और पूजा करना शामिल है।"
- "नीम संवत्सरारम्भ के दिन खाया जाता है।"
- "पञ्चाङ्ग-श्रवण वह पर्व है जिसमें इस दिन पञ्चाङ्ग सुना जाता है।"

Return ONLY valid JSON:
{
    "scratchpad": "<identificaiton of the properties of the entities that describe it>",
    "description": "<single Hindi sentence>"
}
"""

    for chapter_idx in range(10):
        # Read chapter text
        with open(f"text_files/chapter_{chapter_idx}.txt", "r", encoding="utf-8") as f:
            chapter_text = f.read()

        # Load entity extraction outputs
        with open(f"entity_output_files/chapter_{chapter_idx}_output.json", "r", encoding="utf-8") as f:
            old_file = json.load(f)
        with open(f"entity_output_files_redone/chapter_{chapter_idx}_output.json", "r", encoding="utf-8") as f:
            new_file = json.load(f)

        # Build mapping: entity -> set of sentences (only from this chapter)
        chapter_entity_sentences = defaultdict(set)

        for file in (old_file, new_file):
            for group in file:
                for obj in group["extracted_entities"]:
                    sentence = obj["sentence"]
                    for entity in obj["entities"]:
                        chapter_entity_sentences[entity].add(sentence)

        # Generate description for each unseen entity in this chapter
        for entity, sentences in chapter_entity_sentences.items():
            if entity in processed_entities:
                continue
            if entity not in entity_to_id:
                print(f"Warning: Entity '{entity}' not found in entity_id_dict.json. Skipping.")
                continue

            # Prepare user query
            sentences_text = "\n".join(sentences)
            entity_id = entity_to_id[entity]
            user_query = (
                f"Entity: {entity}\n\n"
                f"Chapter text:\n{chapter_text}\n\n"
                # f"Sentences where '{entity}' appears:\n{sentences_text}\n\n"
                f"Generate a one-sentence description which identifies what that entity is or means."
            )

            try:
                response = call_llm(
                    system_prompt=system_prompt,
                    user_query=user_query,
                    model_name="Qwen/Qwen2.5-14B-Instruct",
                    max_tokens=512,          # description is short
                    response_model=EntityDescription
                )
                # response is a JSON string, parse it
                parsed = json.loads(response)
                description = parsed.get("description", "")
                if description:
                    entity_info[entity_id]["description"] = description
                    processed_entities.add(entity)
                    print(f"Generated description for '{entity}': {description}")
                else:
                    print(f"Empty description returned for '{entity}'")
            except Exception as e:
                print(f"Error processing entity '{entity}' in chapter {chapter_idx}: {e}")

        # Save after every chapter to avoid losing progress
        with open("json_files/entity_info.json", "w", encoding="utf-8") as f:
            json.dump(entity_info, f, ensure_ascii=False, indent=2)

    print("All descriptions generated and saved.")

if __name__ == "__main__":
    main()
    
    

Generated description for 'संवत्सरारम्भ': संवत्सरारम्भ वर्ष की शुरुआत का दिन है।
Generated description for 'सूर्योदय': सूर्योदय दिन की शुरुआत का संकेत है।
Generated description for 'प्रतिपदा': प्रतिपदा वर्ष की शुरुआत का दिन है जिसमें विभिन्न धार्मिक और वैज्ञानिक कार्य किए जाते हैं।
Generated description for 'दिन': दिन सूर्य की चक्ररूपी गति का एक अंश है जो कालीन कार्यों को मापने के लिए उपयोग किया जाता है।
Generated description for 'अधिक मास': अधिक मास चंद्रमावादी कैलेंडर में चार पूरे चंद्रमांसों के बाद एक अतिरिक्त दिन या दिनों को शामिल करने वाला मास है।
Generated description for 'चैत्र शुक्ल प्रतिपदा': चैत्र शुक्ल प्रतिपदा हरिवासंवत्सर की शुरुआत का दिन है।
Generated description for 'वर्ष': वर्ष ऋतुओं के परिवर्तन को दर्शाने वाला एक समय कालावयव है।
Generated description for 'घर': घर परिवार के लिए आश्रय और सुरक्षा प्रदान करने वाला स्थान है।
Generated description for 'ध्वजा': ध्वजा ऐश्वर्य और विजय का सूचक है।
Generated description for 'पञ्चाङ्ग-श्रवण': पञ्चाङ्ग-श्रवण एक प्रक्रिया है जिसमें 